## Interactive UI — two-year comparison

Drag the **season** slider. **Spring / Summer:** **2024 vs 2025**; **Autumn / Winter:** **2023 vs 2024**. All seasons use the **same lambda 60-75 redistributed scale** and **yellow to red** colormap (same as Spring). English only.


In [1]:
%pip -q install pandas numpy matplotlib scipy ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm, patheffects as pe
from matplotlib.colors import LinearSegmentedColormap, Normalize, PowerNorm, to_rgb
from scipy.optimize import minimize_scalar, minimize
from scipy.special import gammaln, logsumexp

import ipywidgets as widgets
from IPython.display import display

REPO = Path(".").resolve()
CSV_PATH = REPO / "eda_outputs" / "boar_detections_best.csv"
OUT_PLOTS = REPO / "eda_outputs" / "plots" / "boar_rn_grid_ui"
OUT_PLOTS.mkdir(parents=True, exist_ok=True)

N_COLS = 15
N_ROWS = 10
COL_LETTERS = [chr(ord("A") + i) for i in range(N_COLS)]

plt.rcParams.update({
    "figure.facecolor": "#e8e4dc",
    "axes.facecolor": "#f5f1ea",
    "axes.edgecolor": "#6f6a62",
    "axes.labelcolor": "#1a1814",
    "axes.titleweight": "600",
    "axes.titlesize": 13,
    "axes.labelsize": 10.5,
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Liberation Sans", "Arial", "Helvetica", "sans-serif"],
})
plt.rcParams["axes.unicode_minus"] = False

assert CSV_PATH.is_file(), f"CSV not found: {CSV_PATH}"

_STATION_RE = re.compile(r"^\s*([A-Oa-o])(\d+)")


def station_to_indices(station_id: str) -> tuple[int, int] | None:
    if station_id is None or (isinstance(station_id, float) and np.isnan(station_id)):
        return None
    s = str(station_id).strip()
    m = _STATION_RE.match(s)
    if not m:
        return None
    col = ord(m.group(1).upper()) - ord("A")
    row_num = int(m.group(2))
    if not (1 <= row_num <= N_ROWS and 0 <= col < N_COLS):
        return None
    return col, row_num - 1


def cell_coord(col: int, row: int) -> str:
    return f"{COL_LETTERS[col]}{row + 1}"


def assign_season_label_year(calendar_year: int, month: int) -> tuple[str, int] | None:
    if month in (3, 4, 5):
        return "spring", calendar_year
    if month in (6, 7, 8):
        return "summer", calendar_year
    if month in (9, 10, 11):
        return "autumn", calendar_year
    if month == 12:
        return "winter", calendar_year
    if month in (1, 2):
        return "winter", calendar_year - 1
    return None


SEASON_KEYS = ("spring", "summer", "autumn", "winter")
SEASON_EN = {
    "spring": "Spring (Mar-May)",
    "summer": "Summer (Jun-Aug)",
    "autumn": "Autumn (Sep-Nov)",
    "winter": "Winter (Dec-Feb+1)",
}


def log_binom(J: int, y: int) -> float:
    if y < 0 or y > J:
        return -np.inf
    return gammaln(J + 1) - gammaln(y + 1) - gammaln(J - y + 1)


def loglik_rn_single(lam: float, r: float, y: int, J: int, n_max: int = 120) -> float:
    if J <= 0 or y < 0 or y > J:
        return -np.inf
    lam = max(lam, 1e-12)
    r = min(max(r, 1e-6), 1 - 1e-6)
    log_b = log_binom(J, y)
    ns = np.arange(0, n_max + 1, dtype=float)
    log_pois = -lam + ns * np.log(lam) - gammaln(ns + 1)
    log_omr = np.log1p(-r)
    log_bin = np.empty_like(ns)
    log_bin[ns == 0] = 0.0 if y == 0 else -np.inf
    mask = ns > 0
    pn = -np.expm1(ns[mask] * log_omr)
    pn = np.clip(pn, 1e-15, 1.0 - 1e-15)
    log_bin[mask] = log_b + y * np.log(pn) + (J - y) * np.log1p(-pn)
    tot = log_pois + log_bin
    if not np.any(np.isfinite(tot)):
        return -np.inf
    return float(logsumexp(tot[np.isfinite(tot)]))


def mle_lambda(y: int, J: int, r: float) -> float:
    if J <= 0:
        return 0.0

    def neg_ll(l):
        return -loglik_rn_single(float(l), r, y, J)

    res = minimize_scalar(neg_ll, bounds=(1e-8, 120.0), method="bounded")
    return float(max(res.x, 1e-8))


def neg_loglik_r(r: float, lambdas: np.ndarray, ys: np.ndarray, Js: np.ndarray) -> float:
    r = float(np.clip(r, 1e-5, 1 - 1e-5))
    total = 0.0
    for lam, y, J in zip(lambdas, ys, Js):
        if J <= 0:
            continue
        ll = loglik_rn_single(float(lam), r, int(y), int(J))
        if not np.isfinite(ll):
            return 1e12
        total -= ll
    return total


def fit_rn_alternating(ys: np.ndarray, Js: np.ndarray, n_iter: int = 8) -> tuple[float, np.ndarray]:
    mask = Js > 0
    ys = ys[mask].astype(int)
    Js = Js[mask].astype(int)
    if len(Js) == 0:
        return float("nan"), np.array([])
    r = 0.25
    lambdas = np.array([mle_lambda(int(y), int(J), r) for y, J in zip(ys, Js)])
    for _ in range(n_iter):
        for i, (y, J) in enumerate(zip(ys, Js)):
            lambdas[i] = mle_lambda(int(y), int(J), r)
        res = minimize(
            lambda x: neg_loglik_r(x[0], lambdas, ys, Js),
            x0=[r],
            bounds=[(0.05, 0.95)],
            method="L-BFGS-B",
        )
        r = float(np.clip(res.x[0], 0.05, 0.95))
    return r, lambdas


def aggregate_cells(d: pd.DataFrame) -> pd.DataFrame:
    return d.groupby(["col", "row"], as_index=False).agg(
        J=("filename", "count"),
        y=("detected", "sum"),
        raw_boars=("boar_count", "sum"),
    )


def build_grids(d: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, float]:
    agg = aggregate_cells(d)
    lam_grid = np.full((N_ROWS, N_COLS), np.nan, dtype=float)
    raw_grid = np.zeros((N_ROWS, N_COLS), dtype=float)
    if agg.empty:
        return lam_grid, raw_grid, float("nan")
    ys = agg["y"].to_numpy()
    Js = agg["J"].to_numpy()
    r_hat, lambdas = fit_rn_alternating(ys, Js)
    for (_, row), lam in zip(agg.iterrows(), lambdas):
        c, rr = int(row["col"]), int(row["row"])
        lam_grid[rr, c] = lam
    for _, row in agg.iterrows():
        c, rr = int(row["col"]), int(row["row"])
        raw_grid[rr, c] = float(row["raw_boars"])
    return lam_grid, raw_grid, r_hat

In [3]:
df = pd.read_csv(CSV_PATH)
df["ts_final"] = pd.to_datetime(df["ts_final"], errors="coerce")
df = df.dropna(subset=["station_id", "ts_final"]).copy()
if "filename" not in df.columns:
    df["filename"] = [str(i) for i in df.index]
else:
    _m = df["filename"].isna()
    if _m.any():
        df.loc[_m, "filename"] = [f"__row_{i}" for i in df.index[_m]]
df["station_id"] = df["station_id"].astype(str).str.strip()
df["max_conf"] = pd.to_numeric(df["max_conf"], errors="coerce")
df = df[df["max_conf"] > 0.5].copy()

_coord = df["station_id"].map(station_to_indices)
df = df.loc[_coord.notna()].copy()
df["col"] = [t[0] for t in _coord.loc[df.index]]
df["row"] = [t[1] for t in _coord.loc[df.index]]

df["year"] = df["ts_final"].dt.year.astype(int)
df["month"] = df["ts_final"].dt.month.astype(int)
df["boar_count"] = pd.to_numeric(df["boar_count"], errors="coerce").fillna(0).astype(int)
df["detected"] = (df["boar_count"] > 0).astype(int)
_pairs = [assign_season_label_year(int(y), int(m)) for y, m in zip(df["year"], df["month"])]
df["season"] = [p[0] if p else None for p in _pairs]
df["season_year"] = [p[1] if p else pd.NA for p in _pairs]
df = df.dropna(subset=["season", "season_year"]).copy()
df["season_year"] = df["season_year"].astype(int)

print(f"Rows after max_conf > 0.5 and grid parse: {len(df)}")
CACHE: dict[tuple[int, str], tuple[np.ndarray, np.ndarray, float, int]] = {}
for yr in (2023, 2024, 2025):
    for sk in SEASON_KEYS:
        sub = df[(df["season_year"] == yr) & (df["season"] == sk)]
        CACHE[(yr, sk)] = (*build_grids(sub), len(sub))

print("CSV rows by (season_year, season) — sample:")
print(df.groupby(["season_year", "season"]).size().unstack(fill_value=0).loc[[2023, 2024, 2025], :])

Rows after max_conf > 0.5 and grid parse: 5512
CSV rows by (season_year, season) — sample:
season       autumn  spring  summer  winter
season_year                                
2023             66       0       0     367
2024           1201      26       9    1543
2025              0    1089     961       0


In [4]:
SPRING_LAMBDA_VMIN = 60.0
SPRING_LAMBDA_VMAX = 75.0


def _cmap_spring_redistributed() -> LinearSegmentedColormap:
    """Linear scale: pure yellow at 60 -> orange -> light red -> deep blood red at 75."""
    return LinearSegmentedColormap.from_list(
        "spring_lambda_60_75",
        ["#FFFF00", "#FFB000", "#FF6B1A", "#E32626", "#8B0000", "#3D0000"],
        N=256,
    )


def _cmap_density():
    """Reversed inferno: darker hue = higher density lambda."""
    try:
        c = plt.colormaps["inferno_r"].copy()
    except (AttributeError, KeyError):
        c = cm.get_cmap("inferno_r").copy()
    c.set_bad("#d4cfc6")
    return c


def _norm_from_lambdas(
    *grids: np.ndarray,
    hi_pct: float = 78.0,
    gamma: float = 0.38,
) -> Normalize:
    """Tighter vmax (~P78) + lower gamma: stretch scale so steps read clearly."""
    parts = [g[np.isfinite(g)].ravel() for g in grids if np.any(np.isfinite(g))]
    if not parts:
        return Normalize(0.0, 1.0)
    stacked = np.concatenate(parts)
    if stacked.size == 0:
        return Normalize(0.0, 1.0)
    hi = float(np.percentile(stacked, hi_pct))
    mx = float(np.nanmax(stacked))
    vmax = max(hi, mx * 0.09, 1e-6)
    return PowerNorm(gamma=gamma, vmin=0.0, vmax=vmax, clip=True)


def _cell_label_lines(coord: str, lam: float, raw: float) -> str:
    lam_s = f"λ={lam:.2f}" if np.isfinite(lam) else "λ=n/a"
    return f"{coord}\n{lam_s}\nn={int(raw)}"


def _text_color(norm: Normalize, cmap, val: float) -> str:
    if not np.isfinite(val):
        return "#1a1814"
    vc = float(np.clip(val, norm.vmin, norm.vmax))
    t = float(norm(vc))
    rgb = to_rgb(cmap(t))
    lum = 0.2126 * rgb[0] + 0.7152 * rgb[1] + 0.0722 * rgb[2]
    return "#0d0b09" if lum > 0.5 else "#fffef9"


def plot_density_heatmap(
    lam_grid: np.ndarray,
    raw_grid: np.ndarray,
    *,
    season_key: str,
    season_year: int,
    r_hat: float,
    n_csv: int,
    norm: Normalize | None = None,
    figsize: tuple[float, float] = (9.6, 7.35),
) -> plt.Figure:
    cmap = _cmap_density()
    if norm is None:
        norm = _norm_from_lambdas(lam_grid)

    fig, ax = plt.subplots(figsize=figsize, dpi=120)
    im = ax.imshow(np.ma.masked_invalid(lam_grid), origin="lower", aspect="equal", cmap=cmap, norm=norm)

    for rr in range(N_ROWS):
        for cc in range(N_COLS):
            lam = lam_grid[rr, cc]
            raw = raw_grid[rr, cc]
            if not np.isfinite(lam) and raw <= 0:
                continue
            coord = cell_coord(cc, rr)
            tc = _text_color(norm, cmap, lam if np.isfinite(lam) else 0.0)
            txt = ax.text(
                cc,
                rr,
                _cell_label_lines(coord, lam, raw),
                ha="center",
                va="center",
                fontsize=7.0,
                color=tc,
                linespacing=1.1,
                fontweight="600",
            )
            stroke = "#ffffffd0" if tc.startswith("#0d") else "#1a151090"
            txt.set_path_effects([pe.withStroke(linewidth=2.0, foreground=stroke)])

    ax.set_xticks(np.arange(N_COLS))
    ax.set_xticklabels(COL_LETTERS, fontweight="600")
    ax.set_xlabel("Column A to O (left to right)", fontsize=10.5, fontweight="600")
    ax.set_yticks(np.arange(N_ROWS))
    ax.set_yticklabels([str(i + 1) for i in range(N_ROWS)], fontweight="600")
    ax.set_ylabel("Row 1 to 10 (bottom to top)", fontsize=10.5, fontweight="600")
    ax.set_xlim(-0.5, N_COLS - 0.5)
    ax.set_ylim(-0.5, N_ROWS - 0.5)
    for spine in ax.spines.values():
        spine.set_linewidth(1.2)

    sn = SEASON_EN[season_key]
    rtxt = f"r_hat={r_hat:.3f}" if np.isfinite(r_hat) else "r_hat=n/a"
    ax.set_title(
        f"{sn}  |  season-year {season_year}\nRoyle-Nichols density λ — darker = higher  |  {rtxt}  |  CSV rows: {n_csv}",
        fontsize=11.5,
        pad=12,
    )
    if n_csv == 0:
        ax.text(
            0.5,
            0.5,
            "No CSV rows in this season window",
            transform=ax.transAxes,
            ha="center",
            va="center",
            fontsize=11,
            color="#504c46",
            fontweight="600",
        )

    cbar = fig.colorbar(im, ax=ax, fraction=0.055, pad=0.028, aspect=28)
    cbar.ax.tick_params(labelsize=10)
    cbar.set_label(
        "Density λ — darker = higher (PowerNorm; vmax ~ P78, γ=0.38)",
        fontsize=10.2,
    )
    fig.text(
        0.5,
        0.018,
        "Cell labels: coordinate | λ = density (darker = higher) | n = wild boar count sum",
        ha="center",
        fontsize=8.2,
        color="#504c46",
    )
    fig.tight_layout(rect=(0, 0.038, 1, 1))
    return fig


def _dual_norm(a: np.ndarray, b: np.ndarray) -> Normalize:
    return _norm_from_lambdas(a, b)


def _paint_axis(ax, lam: np.ndarray, raw: np.ndarray, norm: Normalize, cmap, title: str, r_hat: float):
    im = ax.imshow(np.ma.masked_invalid(lam), origin="lower", aspect="equal", cmap=cmap, norm=norm)
    for rr in range(N_ROWS):
        for cc in range(N_COLS):
            l = lam[rr, cc]
            rw = raw[rr, cc]
            if not np.isfinite(l) and rw <= 0:
                continue
            coord = cell_coord(cc, rr)
            tc = _text_color(norm, cmap, l if np.isfinite(l) else 0.0)
            txt = ax.text(
                cc,
                rr,
                _cell_label_lines(coord, l, rw),
                ha="center",
                va="center",
                fontsize=6.5,
                color=tc,
                linespacing=1.06,
                fontweight="600",
            )
            stroke = "#ffffffd0" if tc.startswith("#0d") else "#1a151090"
            txt.set_path_effects([pe.withStroke(linewidth=1.8, foreground=stroke)])
    ax.set_xticks(np.arange(N_COLS))
    ax.set_xticklabels(COL_LETTERS, fontweight="600")
    ax.set_yticks(np.arange(N_ROWS))
    ax.set_yticklabels([str(i + 1) for i in range(N_ROWS)], fontweight="600")
    ax.set_xlabel("Column A to O", fontsize=9.5, fontweight="600")
    ax.set_ylabel("Row 1 to 10 (bottom to top)", fontsize=9.5, fontweight="600")
    rtxt = f"r_hat={r_hat:.3f}" if np.isfinite(r_hat) else "no fit"
    ax.set_title(f"{title}\n{rtxt}", fontsize=11)
    return im

## Static PNG exports

Writes under `eda_outputs/plots/boar_rn_grid_ui/`.


In [5]:
# 2024: four seasons
for sk in SEASON_KEYS:
    lam, raw, rh, nrows = CACHE[(2024, sk)]
    fig = plot_density_heatmap(lam, raw, season_key=sk, season_year=2024, r_hat=rh, n_csv=nrows)
    fig.savefig(OUT_PLOTS / f"density_2024_{sk}.png", dpi=200, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)

# 2025: spring and summer only
for sk in ("spring", "summer"):
    lam, raw, rh, nrows = CACHE[(2025, sk)]
    fig = plot_density_heatmap(lam, raw, season_key=sk, season_year=2025, r_hat=rh, n_csv=nrows)
    fig.savefig(OUT_PLOTS / f"density_2025_{sk}.png", dpi=200, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)

# 2023: autumn and winter only
for sk in ("autumn", "winter"):
    lam, raw, rh, nrows = CACHE[(2023, sk)]
    fig = plot_density_heatmap(lam, raw, season_key=sk, season_year=2023, r_hat=rh, n_csv=nrows)
    fig.savefig(OUT_PLOTS / f"density_2023_{sk}.png", dpi=200, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)

print("Saved:", OUT_PLOTS)

Saved: /home/chengwei/boartraining/HackMITChina2026/eda_outputs/plots/boar_rn_grid_ui


## Spring 2024 vs 2025 — redistributed infographic (export)

Linear **λ** scale **60–75** (yellow → blood red), matching the Spring slider view. Run to save a high-resolution PNG.


In [6]:
label, y_left, y_right = ("2024 (left) vs 2025 (right)", 2024, 2025)
lam_l, raw_l, r_l, n_l = CACHE[(y_left, sk)]
lam_r, raw_r, r_r, n_r = CACHE[(y_right, sk)]
cmap = _cmap_spring_redistributed()
cmap.set_bad("#c9c4bc")
norm = Normalize(vmin=SPRING_LAMBDA_VMIN, vmax=SPRING_LAMBDA_VMAX, clip=True)

fig, axes = plt.subplots(1, 2, figsize=(16.4, 7.15), facecolor="#e8e4dc")
fig.suptitle(
    "Spring (Mar-May) – side-by-side density λ (red = highest, yellow = lowest high) (Redistributed)\n" + label,
    fontsize=13.5,
    fontweight="700",
    y=1.01,
    color="#1a1814",
)
_paint_axis(axes[0], lam_l, raw_l, norm, cmap, f"Season-year {y_left}  |  CSV rows: {n_l}", r_l)
_paint_axis(axes[1], lam_r, raw_r, norm, cmap, f"Season-year {y_right}  |  CSV rows: {n_r}", r_r)
fig.subplots_adjust(bottom=0.20, top=0.82, wspace=0.16)
cax = fig.add_axes([0.14, 0.095, 0.72, 0.042])
sm = cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cb = fig.colorbar(sm, cax=cax, orientation="horizontal")
cb.ax.tick_params(labelsize=11)
cb.set_ticks([60, 65, 70, 75])
cb.set_ticklabels(["60", "65", "70", "75"])
cb.set_label("")
fig.text(
    0.5,
    0.045,
    "Density λ – yellow = low density (marked at 60), red = high density (marked at 75) "
    "(Redistributed 60-75 range for high contrast).",
    ha="center",
    fontsize=10.2,
    color="#1a1814",
    fontweight="600",
)
out = OUT_PLOTS / "spring_mar_may_lambda_redistributed_2024_2025.png"
fig.savefig(out, dpi=320, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.close(fig)
print("Saved:", out)


Saved: /home/chengwei/boartraining/HackMITChina2026/eda_outputs/plots/boar_rn_grid_ui/spring_mar_may_lambda_redistributed_2024_2025.png


## Interactive UI — two-year comparison

Drag the **season** slider. **Spring / Summer:** **2024 vs 2025**; **Autumn / Winter:** **2023 vs 2024**. All seasons use the **same lambda 60-75 redistributed scale** and **yellow to red** colormap (same as Spring). English only.


In [7]:
SEASON_ORDER = ["spring", "summer", "autumn", "winter"]
PAIR_LABEL = {
    "spring": ("2024 (left) vs 2025 (right)", 2024, 2025),
    "summer": ("2024 (left) vs 2025 (right)", 2024, 2025),
    "autumn": ("2023 (left) vs 2024 (right)", 2023, 2024),
    "winter": ("2023 (left) vs 2024 (right)", 2023, 2024),
}


def plot_season_compare(season_key: str):
    sk = season_key
    label, y_left, y_right = PAIR_LABEL[sk]
    lam_l, raw_l, r_l, n_l = CACHE[(y_left, sk)]
    lam_r, raw_r, r_r, n_r = CACHE[(y_right, sk)]

    # Same redistributed 60–75 linear scale + yellow→red map as Spring (all seasons)
    cmap = _cmap_spring_redistributed()
    cmap.set_bad("#c9c4bc")
    norm = Normalize(vmin=SPRING_LAMBDA_VMIN, vmax=SPRING_LAMBDA_VMAX, clip=True)

    fig, axes = plt.subplots(1, 2, figsize=(16.4, 7.15), facecolor="#e8e4dc")
    fig.suptitle(
        f"{SEASON_EN[sk]} – side-by-side density λ (red = highest, yellow = lowest high) (Redistributed)\n"
        + label,
        fontsize=13.5,
        fontweight="700",
        y=1.01,
        color="#1a1814",
    )
    _paint_axis(axes[0], lam_l, raw_l, norm, cmap, f"Season-year {y_left}  |  CSV rows: {n_l}", r_l)
    _paint_axis(axes[1], lam_r, raw_r, norm, cmap, f"Season-year {y_right}  |  CSV rows: {n_r}", r_r)

    fig.subplots_adjust(bottom=0.20, top=0.82, wspace=0.16)
    cax = fig.add_axes([0.14, 0.095, 0.72, 0.042])

    sm = cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cb = fig.colorbar(sm, cax=cax, orientation="horizontal")
    cb.ax.tick_params(labelsize=11)
    cb.set_ticks([60, 65, 70, 75])
    cb.set_ticklabels(["60", "65", "70", "75"])
    cb.set_label("")
    fig.text(
        0.5,
        0.045,
        "Density λ – yellow = low density (marked at 60), red = high density (marked at 75) "
        "(Redistributed 60-75 range; same scale for all seasons).",
        ha="center",
        fontsize=10.2,
        color="#1a1814",
        fontweight="600",
    )
    plt.show()


season_slider = widgets.SelectionSlider(
    options=[(SEASON_EN[k], k) for k in SEASON_ORDER],
    value="spring",
    description="Season:",
    style={"description_width": "4.5rem"},
    layout=widgets.Layout(width="560px"),
)

caption = widgets.HTML(
    value="<div style='font-size:13px;color:#2d2924;margin:4px 0 8px 0;'><b>Spring (Mar-May)</b> — left: <b>2024</b>, right: <b>2025</b></div>"
)


def _caption_for(sk: str) -> str:
    _, a, b = PAIR_LABEL[sk]
    return (
        f"<div style='font-size:13px;color:#2d2924;margin:4px 0 8px 0;'>"
        f"<b>{SEASON_EN[sk]}</b> — left: <b>{a}</b>, right: <b>{b}</b></div>"
    )


def _on_season(change):
    caption.value = _caption_for(change["new"])


season_slider.observe(_on_season, names="value")
_on_season({"new": season_slider.value})

ui = widgets.interactive_output(plot_season_compare, {"season_key": season_slider})

display(
    widgets.VBox(
        [
            widgets.HTML(
                "<p style='margin:0 0 6px 0;color:#444;max-width:860px;line-height:1.5;font-size:13px'>"
                "Drag the slider: <b>Spring / Summer</b> = <b>2024 vs 2025</b>; "
                "<b>Autumn / Winter</b> = <b>2023 vs 2024</b>. "
                "<b>All seasons</b> share the same <b>lambda 60-75</b> yellow-to-red scale. "
                "English labels on figures.</p>"
            ),
            season_slider,
            caption,
            ui,
        ]
    )
)